# Упражнение 9.1

Проверим, почему пример с нарастающей суммой хуже работает для апериодических данных Facebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkdsp import SawtoothSignal, Wave, zero_pad, decorate

Сначала повторим идею главы на периодическом пилообразном сигнале.

In [ ]:
in_wave = SawtoothSignal(freq=50).make_wave(duration=0.1, framerate=44100)
in_wave.unbias()

out_wave = in_wave.cumsum()
out_wave.unbias()

in_wave.plot(label='sawtooth')
out_wave.plot(label='cumsum')
decorate(xlabel='Time (s)')

Для периодического сигнала фильтр `cumsum` можно получить как обратный фильтр к `diff`.

In [ ]:
def make_filter(window, wave):
    padded = zero_pad(window, len(wave))
    wave = Wave(padded, framerate=wave.framerate)
    return wave.make_spectrum()


diff_window = np.array([1.0, -1.0])
diff_filter = make_filter(diff_window, in_wave)

cumsum_filter = diff_filter.copy()
cumsum_filter.hs[1:] = 1 / cumsum_filter.hs[1:]
cumsum_filter.hs[0] = 0

Сравним обычную нарастающую сумму и результат фильтрации в частотной области.

In [ ]:
in_spectrum = in_wave.make_spectrum()
out_wave2 = (in_spectrum * cumsum_filter).make_wave()

out_wave.plot(label='cumsum', alpha=0.7)
out_wave2.plot(label='filtered', alpha=0.7)
decorate(xlabel='Time (s)')

In [ ]:
out_wave.max_diff(out_wave2)

Теперь заменим периодический сигнал апериодическими ценами Facebook.

In [ ]:
df = pd.read_csv('FB_2.csv', header=0, parse_dates=[0])
ys = df['Close']

if len(ys) % 2:
    ys = ys[:-1]

close = Wave(ys, framerate=1)
close.plot()
decorate(xlabel='Time (days)', ylabel='Price ($)')

Возьмем дневные изменения цены и восстановим ряд обычной нарастающей суммой.

In [ ]:
changes = np.diff(close.ys)

if len(changes) % 2:
    changes = changes[:-1]

change = Wave(changes, framerate=1)
expected = change.cumsum()
expected.unbias()

change.plot()
decorate(xlabel='Time (days)', ylabel='Price change ($)')

Применим тот же частотный фильтр `cumsum` к апериодическим изменениям.

In [ ]:
diff_filter = make_filter(diff_window, change)

cumsum_filter = diff_filter.copy()
cumsum_filter.hs[1:] = 1 / cumsum_filter.hs[1:]
cumsum_filter.hs[0] = 0

filtered = (change.make_spectrum() * cumsum_filter).make_wave()

На графике видно расхождение: частотный способ считает сигнал периодическим и создает эффект склейки концов.

In [ ]:
expected.plot(label='ordinary cumsum', alpha=0.7)
filtered.plot(label='spectral cumsum', alpha=0.7)
decorate(xlabel='Time (days)', ylabel='Centered price ($)')

In [ ]:
expected.max_diff(filtered)

Вывод: на апериодических данных Facebook спектральная нарастающая сумма дает заметную ошибку, потому что DFT неявно предполагает периодическое продолжение сигнала.